# Explore

Some helpful links:

- https://docs.voxel51.com/tutorials/clustering.html
- https://docs.voxel51.com/tutorials/dimension_reduction.html

In [1]:
# dataset basics
dataset_label = 'video-game-screenshots'
dataset_exists = False

# since this can take a long time, we don't want to accidently trigger this
calculate_embeddings = False

# we don't want to do this over and over again
compute_visualisations = False

# folder setup
import os
path_samples = os.path.abspath(os.getcwd()) + '/samples/'

## Install FiftyOne

In [ ]:
!pip uninstall -y opencv-python
!pip install -U fiftyone torch torchvision umap-learn scikit-learn
!fiftyone plugins download https://github.com/jacobmarks/clustering-plugin

## Prepare Dataset

### If dataset is persistent and exists

In [3]:
import fiftyone as fo

if dataset_label in fo.list_datasets():
    dataset_exists = True
    dataset = fo.load_dataset('interfaces')
    session = fo.launch_app(dataset, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


[Open Dataset in FiftyOne](http://localhost:5151/)

### If dataset does not yet exist

In [4]:
import os, csv, json

if not dataset_exists:

    dataset = fo.Dataset(dataset_label)
    dataset.persistent = True
    
    # open the wikidata query file to link wikidata_id in filename and csv 
    with open('query.csv') as games_data:
    
        # load reformated mobygames platforms file to get proper platform label
        with open('platforms_reformatted.json') as platforms_data:
            games_rows = csv.reader(games_data, delimiter=',', quotechar='"')
            games = {}
            platforms = json.load(platforms_data)
    
            # save game title and country of origin in tuple
            for game in games_rows:
                games[game[0].split('/')[-1]] = {
                    'title': game[1],
                    'countries': game[4].replace(',', '|') 
                }
    
            # go through all screenshots in the samples folder
            for path, folders, files in os.walk(path_samples):
    
                sample_count = 0
                
                for screenshot in files:
                    
                    filepath = os.path.join(path, screenshot)
    
                    wikidata_id = screenshot.split('_')[0]
    
                    if wikidata_id in games:
    
                        title = games[wikidata_id]['title']
                        platform = platforms[screenshot.split('_')[3]]['platform_name']
                        years = screenshot.split('_')[1]
                        years = years.split('-')
                        countries = games[wikidata_id]['countries']
        
                        sample = fo.Sample(filepath=filepath)
        
                        sample['wikidata_id'] = wikidata_id
                        sample['title'] = title
                        sample['platform'] = platform
                        sample['years'] = years
                        sample['countries'] = countries
                    
                        dataset.add_sample(sample)

    session = fo.launch_app(dataset, auto=False)

### Calculate Embeddings

In [5]:
import fiftyone.brain as fob
import fiftyone.zoo as foz

if calculate_embeddings:
    resnet101 = foz.load_zoo_model("resnet101-imagenet-torch")
    
    dataset.compute_embeddings(
        resnet101,
        embeddings_field="resnet101_embeddings"
    )

In [6]:
if calculate_embeddings:
    dinov2 = foz.load_zoo_model("dinov2-vitb14-torch")
    
    dataset.compute_embeddings(
        dinov2,
        embeddings_field="dinov2_embeddings"
    )

### Compute Visualisations

#### UMAP
1. default parameters
2. dense visualisation
3. lower minimal distance and less neighbours to break up global structures

- [fiftyone.brain.visualization.UMAPVisualizationConfig](https://docs.voxel51.com/api/fiftyone.brain.visualization.html#fiftyone.brain.visualization.UMAPVisualizationConfig)

#### Resnet101

In [7]:
if compute_visualisations:
    res = fob.compute_visualization(
        dataset,
        embeddings="resnet101_embeddings",
        method="umap",
        brain_key="resnet101_umap_vis"
    )
    
    dataset.set_values("resnet101_umap", res.current_points)
    
    res = fob.compute_visualization(
        dataset,
        embeddings="resnet101_embeddings",
        method="umap",
        brain_key="resnet101_umap_tight_vis",
        min_dist=0.5,
        num_neighbors=100
    )
    
    dataset.set_values("resnet101_umap_tight", res.current_points)
    
    res = fob.compute_visualization(
        dataset,
        embeddings="resnet101_embeddings",
        method="umap",
        brain_key="resnet101_umap_loose_vis",
        min_dist=0.001,
        num_neighbors=5
    )
    
    dataset.set_values("resnet101_umap_loose", res.current_points)

#### Dino v2

In [8]:
if compute_visualisations:
    res = fob.compute_visualization(
        dataset,
        embeddings="dinov2_embeddings",
        method="umap",
        brain_key="dinov2_umap_vis"
    )
    
    dataset.set_values("dinov2_umap", res.current_points)
    
    res = fob.compute_visualization(
        dataset,
        embeddings="dinov2_embeddings",
        method="umap",
        brain_key="dinov2_umap_tight_vis",
        min_dist=0.5,
        num_neighbors=100
    )
    
    dataset.set_values("dinov2_umap_tight", res.current_points)
    
    res = fob.compute_visualization(
        dataset,
        embeddings="dinov2_embeddings",
        method="umap",
        brain_key="dinov2_umap_loose_vis",
        min_dist=0.001,
        num_neighbors=5
    )
    
    dataset.set_values("dinov2_umap_loose", res.current_points)